## Useful ideas:
- Ideia 1
- Ideia 2
- Ideia 3

## Useful Content:
- https://towardsdatascience.com/cook-your-first-u-net-in-pytorch-b3297a844cf3 - Coding an UNET
- https://nn.labml.ai/unet/index.html - Another UNET implementation
- https://towardsdatascience.com/transposed-convolution-demystified-84ca81b4baba - Transposed Convolution (UNET basis)

## Fonts
- unet: https://pypi.org/project/unet/ | https://zenodo.org/record/3522306

In [1]:
import sys
# To install everything, remove the '#' from the line below requirements.txt
# !"{sys.executable}" -m pip install -r requirements.txt

                                              0.0/192.3 MB ? eta -:--:--
                                             0.8/192.3 MB 16.6 MB/s eta 0:00:12
                                             1.4/192.3 MB 15.1 MB/s eta 0:00:13
                                             2.1/192.3 MB 14.7 MB/s eta 0:00:13
                                             2.6/192.3 MB 15.3 MB/s eta 0:00:13
                                             3.4/192.3 MB 14.4 MB/s eta 0:00:14
                                             4.0/192.3 MB 14.2 MB/s eta 0:00:14
                                             4.6/192.3 MB 14.8 MB/s eta 0:00:13
     -                                       5.3/192.3 MB 14.2 MB/s eta 0:00:14
     -                                       6.0/192.3 MB 14.1 MB/s eta 0:00:14
     -                                       6.6/192.3 MB 14.1 MB/s eta 0:00:14
     -                                       7.3/192.3 MB 14.1 MB/s eta 0:00:14
     -                                       7.


[notice] A new release of pip is available: 23.1.2 -> 23.2.1
[notice] To update, run: C:\Users\Hugo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import numpy as np
import pandas as pd
import unet 
import torch
import torch
import torchvision.transforms.functional
from torch import nn


In [4]:
class DoubleConvolution(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.first = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.act1 = nn.ReLU()
        self.second = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.act2 = nn.ReLU()
    def forward(self, x: torch.Tensor):
        x = self.first(x)
        x = self.act1(x)
        x = self.second(x)
        return self.act2(x)

In [3]:
class DownSample(nn.Module):
    def __init__(self):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
    def forward(self, x: torch.Tensor):
        return self.pool(x)

In [5]:

class UpSample(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
    def forward(self, x: torch.Tensor):
        return self.up(x)

In [6]:
class CropAndConcat(nn.Module):
    def forward(self, x: torch.Tensor, contracting_x: torch.Tensor):
        contracting_x = torchvision.transforms.functional.center_crop(contracting_x, [x.shape[2], x.shape[3]])
        x = torch.cat([x, contracting_x], dim=1)
        return x

In [8]:
class UNet(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.down_conv = nn.ModuleList([DoubleConvolution(i, o) for i, o in
                                        [(in_channels, 64), (64, 128), (128, 256), (256, 512)]])
        self.down_sample = nn.ModuleList([DownSample() for _ in range(4)])

        self.middle_conv = DoubleConvolution(512, 1024)
        self.up_sample = nn.ModuleList([UpSample(i, o) for i, o in
                                        [(1024, 512), (512, 256), (256, 128), (128, 64)]])
        self.up_conv = nn.ModuleList([DoubleConvolution(i, o) for i, o in
                                      [(1024, 512), (512, 256), (256, 128), (128, 64)]])
        self.concat = nn.ModuleList([CropAndConcat() for _ in range(4)])
        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)
    def forward(self, x: torch.Tensor):
        pass_through = []
        for i in range(len(self.down_conv)):
            x = self.down_conv[i](x)
            pass_through.append(x)
            x = self.down_sample[i](x)
        x = self.middle_conv(x)
        for i in range(len(self.up_conv)):
            x = self.up_sample[i](x)
            x = self.concat[i](x, pass_through.pop())
            x = self.up_conv[i](x)
        x = self.final_conv(x)
        return x